In [ ]:
import json
with open("/home/vectone/Desktop/cdc-pipeline/ontology/meilisearch_documents (1).json","r") as f:
  docs = json.loads(f)

In [ ]:
import meilisearch
client = meilisearch.Client('http://82.113.92.44:7700/', 'iamgroot')

index = client.index('graph_onto1')

index.add_documents(docs, primary_key='id')


In [ ]:
CONFIG = {
    "knowledge_graph": {
        "uri": "neo4j://192.168.11.25:7688",
        "user": "neo4j",
        "password": "India@321", 
        "db_name": "org1000"
    },
    "meilisearch": {
        "url": "http://192.168.11.156:7701",
        "api_key": "iamgroot"
    }
}


In [ ]:
MEILI_CLIENT = None

def get_meilisearch_client(config: dict = None):
    global MEILI_CLIENT
    if MEILI_CLIENT is None and config:
        url = config.get('url', 'http://192.168.11.156:7701')
        api_key = config.get('api_key', 'iamgroot')
        MEILI_CLIENT = meilisearch.Client(url, api_key)
    return MEILI_CLIENT

# ------------------------------------Meilisearch query execution function------------------------------------------
def search_business_concepts(query: str, limit: int = 4) -> dict:
    try:
        config = CONFIG.get('meilisearch', {})
        client = get_meilisearch_client(config)
        if not client:
            return {"error": "Meilisearch client not initialized. Check CONFIG."}
        
        index = client.index('graph_onto')
        search_results = index.search(query, {
            'limit': limit,
            'hybrid': {
                'embedder': 'default',
                'semanticRatio': 0.5
            }
        })
        
        return {
            "query": query,
            "hits": search_results.get('hits', []),
            "total_hits": search_results.get('estimatedTotalHits', 0)
        }
    except Exception as e:
        return {"error": str(e)}

In [ ]:
client.index("graph_onto").update_settings({
    "embedders": {
        "default": {
            "source": "huggingFace",
            "model": "sentence-transformers/all-MiniLM-L6-v2"
        }
    }
})

In [ ]:
search_business_concepts("what about the billing cycles ?")